[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IgnatiusEzeani/spatial-humanities-2026/blob/sh2026-workshop/workshop/00_setup_and_orientation.ipynb)

# AI and NLP for Spatial Humanities
## 00 - Setup and orientation

This workshop uses **spatio-textual** to explore a recurring question: **what kinds of spatial meaning can different computational methods recover from narrative text, and what must a human still validate?**

Today we will move through manual annotation, rules/resources, contextual NLP, transformer models and optional LLM-assisted structured extraction. The aim is not to crown one method as the winner, but to understand their different capabilities, costs and risks.

### Learning goals
By the end of this notebook you should be able to:
1. explain the workshop's distinction between **location**, **locale** and **sense of place**;
2. install and import the lightweight `spatio-textual` environment;
3. run a first spatial annotation;
4. inspect source offsets, place-linking status and telemetry;
5. recognise that an extracted coordinate is only one representation of spatial evidence.

## 1. Install the workshop branch
The default workshop path uses the lightweight spaCy model so that it can run on a normal Colab CPU. Heavier transformer and LLM sections later in the day will have precomputed fallbacks.

In [1]:
!wget -q https://raw.githubusercontent.com/IgnatiusEzeani/spatial-humanities-2026/sh2026-workshop/workshop/sh2026_setup.py

import sh2026_setup as sh
ctx = sh.setup()

# Bound from the shared context: the cells below were written against these.
repo_dir = ctx.repo
data_dir = ctx.data

print('Workshop environment installed.')


Reusing existing checkout at /home/ezeani/workspace/spatial-humanities-2026
Dependencies already installed in this runtime.



Ready in 0s.
  repo    : /home/ezeani/workspace/spatial-humanities-2026
  commit  : 798f2be
  data    : /home/ezeani/workspace/spatial-humanities-2026/workshop/data
  outputs : /home/ezeani/workspace/spatial-humanities-2026/sh2026_outputs
  route   : CPU only, no API key needed

If this cell failed, put your hand up. Do not re-run it more than once.
Workshop environment installed.


In [2]:
import spacy
import pandas as pd
import spatio_textual

from spatio_textual.utils import Annotator, load_spacy_model, split_into_segments

print('spaCy:', spacy.__version__)
print('spatio-textual imported successfully')

spaCy: 3.8.16
spatio-textual imported successfully


## 2. What counts as spatial information?
The project uses three connected ideas:

- **Location**: named places that may be linked to coordinates, such as *Penrith* or *Pooley Bridge*.
- **Locale**: settings and geographical features such as *road*, *lake*, *hill*, *camp* or *ghetto*. These may be vague, relational or non-mappable.
- **Sense of place**: events, perceptions, memories, sentiments and emotions associated with a place.

A core premise of the workshop is that **Spatial Humanities cannot be reduced to finding place names and putting dots on a map**.

## 3. Our first Lake District example
This short nineteenth-century travel-writing example contains named places, a geographical feature and a quantitative spatial relation.

In [3]:
text = (
    'From Penrith two roads lead to Pooley Bridge, about six miles distant, '
    'which spans the Eamont just at its issue from Ulleswater.'
)
print(text)

From Penrith two roads lead to Pooley Bridge, about six miles distant, which spans the Eamont just at its issue from Ulleswater.


### Before running a model
On paper or with a neighbour, mark anything you consider spatial. Do not worry about using the 'correct' label yet.

Questions:
- Which strings are named places?
- Is **road** spatial information?
- Is **about six miles distant** spatial information?
- What information would be lost if we retained only latitude/longitude coordinates?

## 4. Run a first lightweight annotation
This is deliberately a starting point, not a claim of ground truth. The package combines a spaCy pipeline with project resources and can optionally link place-like entities to coordinates.

In [4]:
nlp = load_spacy_model('en_core_web_sm')
annotator = Annotator(nlp, model_name='en_core_web_sm', link_places=True)
record = annotator.annotate(text, include_text=True, include_verbs=True, include_events=True)

record.keys()

/home/ezeani/workspace/spatial-humanities-2026/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


dict_keys(['entities', 'verb_data', 'event_data', 'error', 'requires_review', 'review_notes', 'text', 'telemetry'])

In [5]:
entity_rows = record.get('entities', [])
display(pd.DataFrame(entity_rows))

,text,label,start_char,end_char,start_token,end_token,place_type,confidence,source,resolution_status,lat,lon,geo_source,geo_confidence,ambiguous,candidates_count,candidates
0,Penrith,PERSON,5,12,1,2,NaN,None,spacy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Pooley Bridge,PERSON,31,44,6,8,NaN,None,spacy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,about six miles,QUANTITY,46,61,9,12,NaN,None,spacy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Eamont,GPE,87,93,17,18,PLACE,None,spacy,unresolved,NaN,NaN,none,0.0,False,0.0,[]
4,Ulleswater,GPE,117,127,23,24,PLACE,None,spacy,unresolved,NaN,NaN,none,0.0,False,0.0,[]


### Read the table critically
For each entity, distinguish:
- **what the text actually says** (`text`, `start_char`, `end_char`);
- **what the annotator labels it as** (`label`, `place_type`);
- **what the resolver proposes** (`resolved_name`, coordinates, ambiguity status);
- **which system produced the output** (`source`).

The original string should never be silently replaced by a normalized or geocoded form.

In [6]:
for ent in entity_rows:
    s, e = ent['start_char'], ent['end_char']
    assert text[s:e] == ent['text']
print('All returned entity spans point back to the source text.')

All returned entity spans point back to the source text.


## 5. Segmentation is a research decision
Long narratives cannot always be processed as one unit. `spatio-textual` provides sentence-safe segmentation with source offsets. Later we will also use Q/A-aware segmentation for oral-history-style transcripts.

In [7]:
longer = text + ' ' + 'Either road may be taken. The traveller continues along the lake.'
segments = split_into_segments(longer, max_chars=90, overlap_chars=0, as_records=True)
display(pd.DataFrame(segments))

,text,segStartChar,segEndChar,segTextCharLength
0,"From Penrith two roads lead to Pooley Bridge, ...",0,128,128
1,Either road may be taken. The traveller contin...,129,194,65


Notice that segmentation changes the context available to later models. In Spatial Humanities, this is not merely an engineering setting: it can change what relationships and interpretations are recoverable.

## 6. Inspect telemetry
The workshop will compare methods not only by extraction quality but also by runtime, model/backend, API dependence and review burden.

In [8]:
display(pd.DataFrame(record.get('telemetry', [])))

,task,backend,provider,model,latency_ms,input_chars,input_tokens_est,output_tokens_est,cost_usd_est,success,error
0,spatial_entity_recognition,spacy,local,en_core_web_sm,251.274,128,32,298,0.0,True,None


## 7. The audit rule for the rest of the day
For every computational output, ask four questions:

1. **What evidence in the source supports this?**
2. **Which method/model produced it?**
3. **What is uncertain, ambiguous or missing?**
4. **What would a human need to inspect before using it in an argument?**

Next: **01 - Manual annotation**, where we deliberately start without a model and examine how human annotation decisions themselves shape the task.